# 12 · Block size and spatial folds: review before modelling (5-class)

Run after the R scripts. Everything here is read from R's outputs, never regenerated, and no model has been fitted yet.

1. **Block size**: the blockCV autocorrelation ranges per depth slice and class, and their median (the block size).
2. **Outer folds**: integrity, class counts per fold, and test-to-nearest-training distances.
3. **Inner folds**: the four remaining outer folds are re-used as inner folds when every class occurs in each of them; otherwise R's fallback inner folds are used.

In [ ]:
import os, sys, json
from pathlib import Path

# 5-class soil model: this line selects config/project_5class.toml (keep it).
os.environ["SIEMREAP_CONFIG"] = "project_5class.toml"
# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner(), "|", cfg["_config_name"])

from siemreap import data, folds as F
from siemreap import data as D, plots as PL
CLS = D.class_info(cfg)["classes"]
NAMES = [c["name"] for c in CLS]
COLOURS = PL.class_colours(CLS)
K = len(NAMES)
AGG = config.stage_dir(cfg, "02_folds_review", "aggregate")
sac = config.stage_dir(cfg, "04_spatial_autocorrelation", "aggregate")
bcv = config.stage_dir(cfg, "04_blockcv_folds", "aggregate")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")

## 1. Block size from the autocorrelation range (blockCV)

In [ ]:
ranges = pd.read_csv(sac / "blockcv_autocor_by_slice_class.csv")
display(ranges.pivot_table(index="slice", columns="class_name", values="range_m", sort=False).round(0))
display(ranges.status.value_counts())
decision = F.block_decision(cfg)
print(json.dumps(decision, indent=1))
display(Image(filename=str(sac / "fig_autocor_ranges.png"), width=640))
print("Block sizes of this experiment:", F.block_sizes(cfg), "-> primary", F.primary_block(cfg), "m")

## 2. Outer folds

In [ ]:
blocks = F.block_sizes(cfg)
outer = {b: F.load_outer(cfg, b) for b in blocks}
for b in blocks:
    chk = F.validate_outer(outer[b], mt, cfg)          # raises on any violation
    per = pd.DataFrame(chk["per_fold"]).rename(columns={f"class_{i}": n for i, n in enumerate(NAMES)})
    print(f"{b} m blocks:", {k: v for k, v in chk.items() if k != "per_fold"})
    display(per)
    if (per[NAMES] == 0).any().any():
        print("NOTE: a test fold lacks a class; its recall is undefined in that fold.")

In [ ]:
rows = []
for b in blocks:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        tr, te = F.split(mt, outer[b], k, "A")
        d = F.test_train_distances(tr, te)
        rows.append({"block_m": b, "fold": k, "test_boreholes": len(d), "train_boreholes": tr.BH_Code.nunique(),
                     "min_m": d.min(), "median_m": d.median(), "max_m": d.max(), "share_below_100m": (d < 100).mean(),
                     "share_below_block": (d < b).mean()})
dist = pd.DataFrame(rows).round(2); display(dist)
dist.to_csv(AGG / "test_to_train_distance_by_fold.csv", index=False)
display(Image(filename=str(bcv / f"fig_cv_distance_{blocks[0]}m.png"), width=560)) if (bcv / f"fig_cv_distance_{blocks[0]}m.png").exists() else None

## 3. Inner folds

In [ ]:
rows = []
for b in blocks:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        tr, _ = F.split(mt, outer[b], k, "A")
        g, a, source = F.inner_groups(tr, outer[b], k, cfg, b)
        rows.append({"block_m": b, "outer_fold": k, "inner_source": source, "all_ok": bool(a.ok.all()),
                     "min_boreholes": a.boreholes.min(), "min_cells": a.cells.min(), "min_class_cells": a.min_class_cells.min()})
adq = pd.DataFrame(rows); display(adq); adq.to_csv(AGG / "inner_fold_adequacy.csv", index=False)
(AGG / "folds_review.json").write_text(json.dumps({"block_decision": decision, "block_sizes": blocks,
                                                   "inner_sources": adq.inner_source.value_counts().to_dict()}, indent=1, default=str))